# Enterprise Search — Activating Knowledge from Annual Reports

**Enterprise search and RAG are not just retrieval.** The hard part is turning messy, chart-heavy documents into AI-ready knowledge while preserving layout, tables, sections, and source locations — then serving grounded answers with citations.

This notebook walks a live pipeline over **five consumer-goods annual reports** (Unilever, Nestlé, P&G, PepsiCo, plus Coca-Cola as a 10-K peer):

1. **Parse** each PDF with layout-aware OCR (`AI_PARSE_DOCUMENT`) — one row per page.
2. **Describe charts** on staged page images with vision (`AI_COMPLETE`) and fold that narrative into the searchable text.
3. **Chunk** the enriched content and index it in a **Cortex Search** service.
4. **Answer questions** with RAG — retrieve top-k chunks, then `AI_COMPLETE` with page-accurate citations.

Everything is declarative Snowflake Cortex SQL on incremental dynamic tables: each AI function runs **once per new file**, not once per refresh.

```
DEMO_ESR_DOCS_STAGE (report PDFs + per-page PNGs)
  -> DEMO_ESR_FILE_LOG        stream + task (event-driven ingest)
  -> DT_DEMO_ESR_PARSED       AI_PARSE_DOCUMENT(LAYOUT, page_split)
  -> DT_DEMO_ESR_PAGES        FLATTEN :pages -> one row per (company, page)
  -> DT_DEMO_ESR_FIGURES      AI_COMPLETE(vision) on page PNGs -> chart narrative
  -> DT_DEMO_ESR_ENRICHED     page text + chart narrative
  -> DT_DEMO_ESR_CHUNK_ARR    SPLIT_TEXT_RECURSIVE_CHARACTER -> array
  -> DT_DEMO_ESR_CHUNKS       FLATTEN -> one row per chunk
  -> DEMO_ESR_SEARCH          CREATE CORTEX SEARCH SERVICE
  -> RAG                      SEARCH_PREVIEW + AI_COMPLETE + citations
```

The hero: a query like *"what does P&G's net-sales-by-segment chart show?"* returns an answer grounded in vision-described chart content — not just body text near the figure.

> **Before running:** this notebook reads objects the pipeline already built, so first run `00_setup.sql`, the sourcing script, `EXECUTE TASK {database}.{schema}.DEMO_ESR_INGEST_TASK`, and `10_pipeline.sql`, and let the dynamic tables and Cortex Search service refresh. Then substitute `{database}` / `{schema}` / `{warehouse}` throughout — they appear in the context cell below and in the fully-qualified Cortex Search service name `{database}.{schema}.DEMO_ESR_SEARCH` used by the retrieval and RAG cells. All other object references resolve against the schema set in the context cell.

In [ ]:
USE SCHEMA {database}.{schema};
USE WAREHOUSE {warehouse};

## 1 · The corpus

Five annual reports in one sector, so cross-filing search is meaningful. Page images are rendered at sourcing time so vision can read charts that `LAYOUT` parsing alone would miss.

In [ ]:
SELECT c.SLUG, c.COMPANY, c.TICKER, c.KIND,
       COUNT(DISTINCT fl.RELATIVE_PATH) AS staged_files
FROM DEMO_ESR_COMPANIES c
LEFT JOIN DEMO_ESR_FILE_LOG fl
  ON fl.RELATIVE_PATH ILIKE 'reports/' || c.SLUG || '.pdf'
  OR fl.RELATIVE_PATH ILIKE 'pages/' || c.SLUG || '/%'
GROUP BY c.SLUG, c.COMPANY, c.TICKER, c.KIND
ORDER BY c.COMPANY;

Every dynamic table is **incremental** (`refresh_mode = INCREMENTAL`), so when new reports land only those files are processed. The two `LATERAL FLATTEN` layers (pages, chunks) and the join layer stay incremental because they flatten **materialized** array columns.

In [ ]:
SHOW DYNAMIC TABLES LIKE 'DT_DEMO_ESR%';
SELECT "name", "refresh_mode", "target_lag", "scheduling_state"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY "name";

In [ ]:
SELECT 'file_log' AS layer, COUNT(*) AS n FROM DEMO_ESR_FILE_LOG
UNION ALL SELECT 'parsed',   COUNT(*) FROM DT_DEMO_ESR_PARSED
UNION ALL SELECT 'pages',    COUNT(*) FROM DT_DEMO_ESR_PAGES
UNION ALL SELECT 'figures',  COUNT(*) FROM DT_DEMO_ESR_FIGURES
UNION ALL SELECT 'enriched', COUNT(*) FROM DT_DEMO_ESR_ENRICHED
UNION ALL SELECT 'chunks',   COUNT(*) FROM DT_DEMO_ESR_CHUNKS
ORDER BY layer;

## 2 · Chart vision — making figures searchable

`DT_DEMO_ESR_FIGURES` runs `AI_COMPLETE` (vision) over each staged page PNG. Pages with no charts return `NO_CHART`; pages with donut charts, bar charts, or financial callouts get a prose narrative that is merged into the chunk text at enrich time.

In [ ]:
SELECT COMPANY,
       COUNT(*)                                        AS pages_with_image,
       COUNT_IF(CHART_NARRATIVE NOT ILIKE 'NO_CHART%') AS pages_with_charts
FROM DT_DEMO_ESR_FIGURES
GROUP BY COMPANY
ORDER BY pages_with_charts DESC;

In [ ]:
SELECT COMPANY, PAGE, LEFT(CHART_NARRATIVE, 300) AS chart_narrative_preview
FROM DT_DEMO_ESR_FIGURES
WHERE CHART_NARRATIVE NOT ILIKE 'NO_CHART%'
ORDER BY COMPANY, PAGE
LIMIT 5;

## 3 · Retrieval — the search index in action

Raw `SEARCH_PREVIEW` hits (no LLM) confirm the index serves relevant passages — including chart narratives when the query targets a visual topic.

In [ ]:
SELECT v.value:COMPANY::STRING AS COMPANY, v.value:PAGE::INT AS PAGE,
       LEFT(v.value:CHUNK::STRING, 220) AS CHUNK_PREVIEW
FROM TABLE(FLATTEN(input => PARSE_JSON(
      SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        '{database}.{schema}.DEMO_ESR_SEARCH',
        '{"query":"foreign exchange currency risk exposure","columns":["CHUNK","COMPANY","PAGE"],"limit":5}'
      ))['results'])) v;

## 4 · RAG with citations

Retrieve top-k chunks, then `AI_COMPLETE` answers using **only** that context. Every claim cites company and page — the `PAGE` attribute on the search service makes citations page-accurate.

### Q1 · Cross-filing risk — which companies flag FX / currency exposure?

In [ ]:
WITH hits AS (
  SELECT v.value:CHUNK::STRING AS CHUNK, v.value:COMPANY::STRING AS SLUG, v.value:PAGE::INT AS PAGE
  FROM TABLE(FLATTEN(input => PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
          '{database}.{schema}.DEMO_ESR_SEARCH',
          '{"query":"Which companies flag foreign-exchange or currency risk as a major exposure?","columns":["CHUNK","COMPANY","PAGE"],"limit":10}'
        ))['results'])) v
),
ctx AS (
  SELECT h.CHUNK, h.PAGE, COALESCE(co.COMPANY, h.SLUG) AS COMPANY
  FROM hits h LEFT JOIN DEMO_ESR_COMPANIES co ON co.SLUG = h.SLUG
)
SELECT AI_COMPLETE('claude-4-sonnet',
  'Answer the question using ONLY the provided context from company annual reports. '
  || 'Cite the company and page for every claim, like (Company, p.N). If the context is '
  || 'insufficient, say so.\n\nQuestion: Which companies flag foreign-exchange or currency '
  || 'risk as a major exposure, and how do they describe it?\n\nContext:\n'
  || LISTAGG(COMPANY || ' (p.' || PAGE || '): ' || CHUNK, '\n---\n')
       WITHIN GROUP (ORDER BY COMPANY, PAGE)
) AS ANSWER
FROM ctx;

### Q2 · Comparative theme — AI, technology, and R&D investment across filings

In [ ]:
WITH hits AS (
  SELECT v.value:CHUNK::STRING AS CHUNK, v.value:COMPANY::STRING AS SLUG, v.value:PAGE::INT AS PAGE
  FROM TABLE(FLATTEN(input => PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
          '{database}.{schema}.DEMO_ESR_SEARCH',
          '{"query":"investment in AI, technology, R&D, digital and productivity","columns":["CHUNK","COMPANY","PAGE"],"limit":12}'
        ))['results'])) v
),
ctx AS (
  SELECT h.CHUNK, h.PAGE, COALESCE(co.COMPANY, h.SLUG) AS COMPANY
  FROM hits h LEFT JOIN DEMO_ESR_COMPANIES co ON co.SLUG = h.SLUG
)
SELECT AI_COMPLETE('claude-4-sonnet',
  'Using ONLY the context from these company annual reports, compare what each company says '
  || 'about investment in AI, technology, R&D, and productivity. Give one short paragraph per '
  || 'company and cite (Company, p.N).\n\nContext:\n'
  || LISTAGG(COMPANY || ' (p.' || PAGE || '): ' || CHUNK, '\n---\n')
       WITHIN GROUP (ORDER BY COMPANY, PAGE)
) AS ANSWER
FROM ctx;

### Q3 · Chart-searchability hero — P&G net sales by segment and geography

The proof point: the answer depends on **chart content** described by vision and folded into chunks. Filter to P&G via the `COMPANY` search attribute.

In [ ]:
WITH hits AS (
  SELECT v.value:CHUNK::STRING AS CHUNK, v.value:COMPANY::STRING AS SLUG, v.value:PAGE::INT AS PAGE
  FROM TABLE(FLATTEN(input => PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
          '{database}.{schema}.DEMO_ESR_SEARCH',
          '{"query":"net sales by business segment and by geographic region chart breakdown percentages","columns":["CHUNK","COMPANY","PAGE"],"filter":{"@eq":{"COMPANY":"pg"}},"limit":8}'
        ))['results'])) v
),
ctx AS (
  SELECT h.CHUNK, h.PAGE, COALESCE(co.COMPANY, h.SLUG) AS COMPANY
  FROM hits h LEFT JOIN DEMO_ESR_COMPANIES co ON co.SLUG = h.SLUG
)
SELECT AI_COMPLETE('claude-4-sonnet',
  'Using ONLY the context, describe what Procter and Gamble net-sales-by-segment and '
  || 'net-sales-by-geography charts show, including the segment and region percentages. '
  || 'Cite the page(s).\n\nContext:\n'
  || LISTAGG(COMPANY || ' (p.' || PAGE || '): ' || CHUNK, '\n---\n')
       WITHIN GROUP (ORDER BY COMPANY, PAGE)
) AS ANSWER
FROM ctx;

## Scale

This demo runs on **five reports**, but nothing about the pipeline is sized to that. It is `INCREMENTAL` end to end: a stream + task lands new files, and every dynamic table refreshes **only on new rows** — so `AI_PARSE_DOCUMENT` and vision run **once per file, ever**. The Cortex Search service refreshes within its `TARGET_LAG`. The same declarative SQL that runs on five reports runs on a portfolio of thousands.

> **Text-only variant:** drop `DT_DEMO_ESR_FIGURES` and the enrich merge; index parsed page text directly. Loses chart-only facts; everything else is unchanged.